# Lab: Generative AI for Data Science
## Scenario 1: Retail Inventory Analysis

Workflow: Prompt → Generated Code → Follow-up Prompts → Final Solution → Discussion.

## 1. Your Prompt

I'm analyzing retail inventory turnover in Python (pandas, matplotlib). My DataFrame `df` has columns: product_id, category, stock_level, last_restock_date, sales_last_30_days, supplier_lead_time (days), unit_cost.

Write code that:

1. Calculates inventory turnover (30-day and annualized).
2. Flags slow-moving items (lowest turnover or very high days of supply).
3. Predicts stockouts: estimated stockout date vs. supplier lead time.
4. Creates visualizations: turnover by category, slow movers, stockout risk.

Requirements: parse dates safely, handle missing values and zero sales (no divide-by-zero), use clear function names and comments.

**Why I set it up this way:** I named the exact columns and libraries so the code matches the data, listed the four tasks separately so none is skipped, and asked for error handling up front because zero sales and bad dates are the usual failures.

## 2. Generated Code (first draft)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample data (replace with your own df)
np.random.seed(42)
n = 40
df = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(n)],
    "category": np.random.choice(["Grocery", "Electronics", "Apparel", "Home"], n),
    "stock_level": np.random.randint(0, 300, n),
    "last_restock_date": pd.Timestamp("2026-11-09") - pd.to_timedelta(np.random.randint(1, 60, n), unit="D"),
    "sales_last_30_days": np.random.randint(0, 250, n),
    "supplier_lead_time": np.random.randint(3, 21, n),
    "unit_cost": np.round(np.random.uniform(2, 120, n), 2),
})

def clean_data(df):
    df = df.copy()
    df["last_restock_date"] = pd.to_datetime(df["last_restock_date"], errors="coerce")
    num_cols = ["stock_level", "sales_last_30_days", "supplier_lead_time", "unit_cost"]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    return df.dropna(subset=num_cols + ["product_id"])

def add_turnover(df):
    df = df.copy()
    cogs = df["sales_last_30_days"] * df["unit_cost"]
    inv_value = (df["stock_level"] * df["unit_cost"]).replace(0, np.nan)
    df["turnover_30d"] = (cogs / inv_value).fillna(0)
    df["turnover_annual"] = df["turnover_30d"] * (365 / 30)
    df["days_since_restock"] = (pd.Timestamp.today().normalize() - df["last_restock_date"]).dt.days
    return df

def flag_slow_movers(df):
    df = df.copy()
    daily_sales = df["sales_last_30_days"] / 30
    df["days_of_supply"] = np.where(daily_sales > 0, df["stock_level"] / daily_sales, np.inf)
    cutoff = df["turnover_30d"].quantile(0.25)
    df["slow_moving"] = (df["turnover_30d"] <= cutoff) | (df["days_of_supply"] > 90)
    return df

def predict_stockouts(df):
    df = df.copy()
    df["stockout_date"] = pd.Timestamp.today().normalize() + pd.to_timedelta(
        df["days_of_supply"].replace(np.inf, 9999), unit="D")
    df["stockout_risk"] = df["days_of_supply"] <= df["supplier_lead_time"]
    return df

def make_plots(df):
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    df.groupby("category")["turnover_annual"].mean().plot.bar(ax=ax[0], color="steelblue")
    ax[0].set_title("Avg Annual Turnover by Category")
    ax[0].set_ylabel("Turnover")

    slow = df[df["slow_moving"]].nsmallest(10, "turnover_30d")
    ax[1].barh(slow["product_id"], slow["turnover_30d"], color="indianred")
    ax[1].set_title("10 Slowest-Moving Items")

    colors = df["stockout_risk"].map({True: "red", False: "green"})
    ax[2].scatter(df["supplier_lead_time"], df["days_of_supply"].clip(upper=200), c=colors)
    ax[2].plot([0, 25], [0, 25], "k--", label="Days of supply = lead time")
    ax[2].set_xlabel("Supplier lead time (days)")
    ax[2].set_ylabel("Days of supply")
    ax[2].set_title("Stockout Risk (red = at risk)")
    ax[2].legend()
    plt.tight_layout()
    plt.show()

result = predict_stockouts(flag_slow_movers(add_turnover(clean_data(df))))
print(result[result["stockout_risk"]][["product_id", "stock_level", "days_of_supply", "stockout_date"]])
make_plots(result)

## 3. Follow-up Prompts
1. "Use today's date dynamically, not a hardcoded date, and handle stock_level = 0."
2. "Add a column showing recommended reorder quantity (lead-time demand plus 20% safety stock)."
3. "Make the charts consistent: same colors, labeled axes, readable on one row."

**Hurdles:** the first draft crashed on zero-sales items (infinite days of supply), and I had to be explicit about how to define turnover because the data has no COGS column (I used sales × unit cost).

## 4. Final Solution (after refinements)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TODAY = pd.Timestamp.today().normalize()   # dynamic date, never hardcoded

# ---- Sample data (replace with your own df) ----
np.random.seed(42)
n = 40
df = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(n)],
    "category": np.random.choice(["Grocery", "Electronics", "Apparel", "Home"], n),
    "stock_level": np.random.randint(0, 300, n),
    "last_restock_date": TODAY - pd.to_timedelta(np.random.randint(1, 60, n), unit="D"),
    "sales_last_30_days": np.random.randint(0, 250, n),
    "supplier_lead_time": np.random.randint(3, 21, n),
    "unit_cost": np.round(np.random.uniform(2, 120, n), 2),
})

def clean_data(df):
    """Parse dates safely, coerce numbers, drop rows missing key fields."""
    df = df.copy()
    df["last_restock_date"] = pd.to_datetime(df["last_restock_date"], errors="coerce")
    num_cols = ["stock_level", "sales_last_30_days", "supplier_lead_time", "unit_cost"]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    return df.dropna(subset=num_cols + ["product_id"])

def add_turnover(df):
    """Turnover = (30-day sales x cost) / (stock x cost). Zero stock -> 0, no crash."""
    df = df.copy()
    cogs = df["sales_last_30_days"] * df["unit_cost"]
    inv_value = (df["stock_level"] * df["unit_cost"]).replace(0, np.nan)
    df["turnover_30d"] = (cogs / inv_value).fillna(0)
    df["turnover_annual"] = df["turnover_30d"] * (365 / 30)
    df["days_since_restock"] = (TODAY - df["last_restock_date"]).dt.days
    return df

def flag_slow_movers(df):
    """Slow = bottom 25% turnover OR more than 90 days of supply."""
    df = df.copy()
    daily_sales = df["sales_last_30_days"] / 30
    df["days_of_supply"] = np.where(daily_sales > 0, df["stock_level"] / daily_sales, np.inf)
    cutoff = df["turnover_30d"].quantile(0.25)
    df["slow_moving"] = (df["turnover_30d"] <= cutoff) | (df["days_of_supply"] > 90)
    return df

def predict_stockouts(df):
    """Stockout risk if stock runs out before a new order could arrive."""
    df = df.copy()
    df["stockout_date"] = TODAY + pd.to_timedelta(df["days_of_supply"].replace(np.inf, 9999), unit="D")
    df["stockout_risk"] = df["days_of_supply"] <= df["supplier_lead_time"]
    # Reorder qty = lead-time demand + 20% safety stock
    df["reorder_qty"] = np.ceil((df["sales_last_30_days"] / 30) * df["supplier_lead_time"] * 1.2).astype(int)
    return df

def make_plots(df):
    """Three consistent, labeled charts on one row."""
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))

    df.groupby("category")["turnover_annual"].mean().sort_values().plot.bar(ax=ax[0], color="steelblue")
    ax[0].set_title("Avg Annual Turnover by Category")
    ax[0].set_xlabel("Category"); ax[0].set_ylabel("Annual turnover (x)")

    slow = df[df["slow_moving"]].nsmallest(10, "turnover_30d")
    ax[1].barh(slow["product_id"], slow["turnover_30d"], color="indianred")
    ax[1].set_title("10 Slowest-Moving Items")
    ax[1].set_xlabel("30-day turnover (x)"); ax[1].set_ylabel("Product")

    colors = df["stockout_risk"].map({True: "red", False: "green"})
    ax[2].scatter(df["supplier_lead_time"], df["days_of_supply"].clip(upper=200), c=colors)
    lim = df["supplier_lead_time"].max() + 2
    ax[2].plot([0, lim], [0, lim], "k--", label="Days of supply = lead time")
    ax[2].set_title("Stockout Risk (red = at risk)")
    ax[2].set_xlabel("Supplier lead time (days)"); ax[2].set_ylabel("Days of supply (capped at 200)")
    ax[2].legend()

    plt.tight_layout()
    plt.show()

# ---- Run the pipeline ----
result = predict_stockouts(flag_slow_movers(add_turnover(clean_data(df))))

print("Items at risk of stocking out:")
display(result[result["stockout_risk"]][["product_id", "category", "stock_level", "days_of_supply", "stockout_date", "reorder_qty"]])

print("Slow-moving items:", int(result["slow_moving"].sum()))
make_plots(result)

## 5. Discussion Answers
*Written for my version of the prompts. Adjust to match your own outputs.*

**1. How did different prompts handle date calculations?**
A vague prompt usually produces code that assumes dates are already datetime objects. My prompt asked for safe parsing, so the code uses `pd.to_datetime(errors="coerce")` and `Timestamp.today()` to compute days since restock and the projected stockout date. The follow-up removed a hardcoded date so the analysis always runs from the current day.

**2. What visualization approaches were suggested?**
A bar chart for turnover by category, a horizontal bar chart for the 10 slowest items, and a scatter plot of days of supply against lead time with a break-even line (red points = at risk). A prompt that doesn't ask for visualizations in detail tends to return a single generic bar chart.

**3. How was error handling implemented?**
`errors="coerce"` for bad dates and numbers, `dropna` for missing key fields, and `replace(0, np.nan)` plus `np.where` to avoid dividing by zero when stock or sales are 0. Without asking for this in the prompt, generated code usually has none of it.

---
## Scenario 2: Website Analytics Debug

**Scenario:** The marketing team reports that the user engagement metrics code is showing impossible results (bounce rates over 100%, negative session times).

**Task:** Debug the calculations, add data validation, implement proper time calculations, and create summary visualizations.

### Problematic Code (given) and sample data

The sample logs below are generated for testing. Each row is one log record for a session, and some rows are deliberately dirty (negative durations, missing devices, duplicates, bad dates) to reproduce the problem. Replace `logs_df` with your real data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Sample logs (replace with your own logs_df) ----
rng = np.random.default_rng(7)
base = pd.Timestamp("2026-10-01", tz="UTC")
rows = []
for i in range(300):
    sid = f"S{i:04d}"
    device = rng.choice(["Desktop", "Mobile", "Tablet"], p=[0.45, 0.45, 0.10])
    n_records = rng.choice([1, 2, 3, 4, 5, 6], p=[0.35, 0.25, 0.18, 0.12, 0.06, 0.04])
    t = base + pd.Timedelta(minutes=int(rng.integers(0, 40 * 24 * 60)))
    for _ in range(n_records):
        dur = int(rng.integers(5, 300))
        rows.append({"session_id": sid, "device_type": device, "page_views": 1,
                     "start_time": t, "end_time": t + pd.Timedelta(seconds=dur), "duration": dur})
        t = t + pd.Timedelta(seconds=dur)
logs_df = pd.DataFrame(rows)

# ---- Inject realistic dirty data ----
dirty = rng.choice(logs_df.index, 44, replace=False)
neg, huge, nodev, badpv, baddate = dirty[:12], dirty[12:20], dirty[20:28], dirty[28:36], dirty[36:44]
logs_df.loc[neg, "end_time"] = logs_df.loc[neg, "start_time"] - pd.Timedelta(seconds=120)   # end before start
logs_df.loc[neg, "duration"] = -120
logs_df.loc[huge, "end_time"] = logs_df.loc[huge, "start_time"] + pd.Timedelta(days=2)       # 2-day "session"
logs_df.loc[huge, "duration"] = 172800
logs_df.loc[nodev, "device_type"] = None
logs_df.loc[badpv, "page_views"] = rng.choice([0, -1], len(badpv))
logs_df["start_time"] = logs_df["start_time"].dt.strftime("%Y-%m-%d %H:%M:%S%z")
logs_df["end_time"] = logs_df["end_time"].dt.strftime("%Y-%m-%d %H:%M:%S%z")
logs_df.loc[baddate, "end_time"] = "not_a_date"
logs_df = pd.concat([logs_df, logs_df.sample(10, random_state=1)], ignore_index=True)       # duplicate rows

# ---- Original problematic code (renamed so it can sit beside the fix) ----
def analyze_user_engagement_buggy(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(
            lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }

    device_metrics = logs_df.groupby('device_type').agg({
        'session_id': 'count',
        'duration': 'mean',
        'page_views': 'sum'
    })

    return metrics, device_metrics

buggy_metrics, buggy_device = analyze_user_engagement_buggy(logs_df)
print("bounce_rate:", round(buggy_metrics["bounce_rate"], 3), "(row-level True/False average, not a session bounce rate)")
print("avg_session_time is a", type(buggy_metrics["avg_session_time"]).__name__,
      "of per-session SUMS, min =", buggy_metrics["avg_session_time"].min(), "seconds (negative!)")
print("pages_per_session is a", type(buggy_metrics["pages_per_session"]).__name__, "(one value per session, not one number)")
display(buggy_device)

## 1. Your Prompt

The marketing team says my user engagement code gives impossible results: bounce rates over 100% and negative session times. I'm using pandas on a logs DataFrame `logs_df` with columns session_id, device_type, page_views, start_time, end_time and duration (in seconds). Each row is one log record, and a session can have several records. *(I pasted the problematic `analyze_user_engagement` function here.)*

Please:

1. Explain each bug in the calculations and why it produces impossible values.
2. Fix the calculations so every metric is computed at the session level.
3. Add data validation (missing values, duplicates, invalid page views, bad or negative durations) and report how many rows were removed.
4. Calculate durations properly from the timestamps, handling bad dates and time zones.
5. Create summary visualizations by device type.

Return clean, commented functions with clear names.

**Why I set it up this way:** I pasted the broken code and described the symptoms so the model had to find the root cause instead of guessing. I asked for an explanation of each bug first so I could check its diagnosis, numbered the requests so none was skipped, and described the columns and what one row means, because "session" and "bounce" are easy to define differently.

## 2. Generated Code (first draft)

In [ ]:
def analyze_user_engagement(logs_df):
    df = logs_df.copy()

    # Basic validation
    df = df.dropna(subset=["session_id", "device_type"])
    df["page_views"] = pd.to_numeric(df["page_views"], errors="coerce")
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    df = df[(df["page_views"] >= 1) & (df["duration"] >= 0)]

    # One row per session (the original code mixed row-level and session-level data)
    sessions = df.groupby("session_id").agg(
        device_type=("device_type", "first"),
        page_views=("page_views", "sum"),
        duration=("duration", "sum"),
    )

    metrics = {
        "bounce_rate": (sessions["page_views"] == 1).mean(),
        "avg_session_time": sessions["duration"].mean(),
        "pages_per_session": sessions["page_views"].mean(),
    }

    device_metrics = sessions.groupby("device_type").agg(
        sessions=("page_views", "count"),
        avg_duration=("duration", "mean"),
        total_page_views=("page_views", "sum"),
    )
    return metrics, device_metrics

def plot_draft(device_metrics):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    device_metrics["sessions"].plot.bar(ax=ax[0])
    ax[0].set_title("Sessions by Device")
    device_metrics["avg_duration"].plot.bar(ax=ax[1])
    ax[1].set_title("Avg Session Duration by Device")
    plt.tight_layout()
    plt.show()

draft_metrics, draft_device = analyze_user_engagement(logs_df)
print(draft_metrics)
display(draft_device)
plot_draft(draft_device)

## 3. Follow-up Prompts

1. "Don't trust the logged `duration` column. Calculate it from `start_time` and `end_time`, handle unparseable dates and time zones, and reject durations that are negative or longer than 24 hours."
2. "Also remove duplicate rows, and return a validation report showing how many rows were removed for each reason."
3. "Add sanity checks so the function fails loudly if bounce rate is outside 0-100% or average session time is negative."
4. "Improve the charts: show bounce rate in percent, session time in minutes, label every axis, and add a duration histogram."

**Hurdles:** the first draft fixed the aggregation but still trusted the logged `duration` column, did not remove duplicates, and silently dropped bad rows without saying how many. I also had to decide what a "bounce" means (I used a session with exactly one page view) and state that each row is a log record, not a session.

## 4. Final Solution (after refinements)

In [ ]:
REQUIRED_COLS = ["session_id", "device_type", "page_views", "start_time", "end_time", "duration"]
MAX_SESSION_SECONDS = 24 * 60 * 60   # anything longer than 24h is treated as bad data

def validate_logs(logs_df):
    """Clean raw logs and report how many rows were removed and why."""
    missing = [c for c in REQUIRED_COLS if c not in logs_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = logs_df.copy()
    report = {"rows_in": len(df)}

    # 1. Duplicate rows
    n = len(df); df = df.drop_duplicates()
    report["duplicates"] = n - len(df)

    # 2. Missing session id or device
    n = len(df); df = df.dropna(subset=["session_id", "device_type"])
    report["missing_id_or_device"] = n - len(df)

    # 3. Page views must be a number >= 1 (NaN fails the comparison and is dropped)
    df["page_views"] = pd.to_numeric(df["page_views"], errors="coerce")
    n = len(df); df = df[df["page_views"] >= 1]
    report["invalid_page_views"] = n - len(df)

    # 4. Proper time calculation: use timestamps (UTC), fall back to logged duration only if they can't be parsed
    start = pd.to_datetime(df["start_time"], errors="coerce", utc=True)
    end = pd.to_datetime(df["end_time"], errors="coerce", utc=True)
    from_timestamps = (end - start).dt.total_seconds()
    logged = pd.to_numeric(df["duration"], errors="coerce")
    df["duration_s"] = from_timestamps.fillna(logged)

    # 5. Reject negative, missing or absurdly long durations
    n = len(df); df = df[df["duration_s"].between(0, MAX_SESSION_SECONDS)]
    report["invalid_duration"] = n - len(df)

    report["rows_out"] = len(df)
    return df, report

def build_sessions(clean_df):
    """Collapse log records into one row per session."""
    return clean_df.groupby("session_id").agg(
        device_type=("device_type", "first"),
        page_views=("page_views", "sum"),
        duration_s=("duration_s", "sum"),
    )

def analyze_user_engagement(logs_df):
    """Return overall metrics, per-device metrics, session table and validation report."""
    clean_df, report = validate_logs(logs_df)
    if clean_df.empty:
        raise ValueError("No valid rows left after validation")

    sessions = build_sessions(clean_df)

    metrics = {
        "total_sessions": len(sessions),
        "bounce_rate_pct": (sessions["page_views"] == 1).mean() * 100,   # 1 page view = bounce
        "avg_session_time_s": sessions["duration_s"].mean(),
        "pages_per_session": sessions["page_views"].mean(),
    }

    # Sanity checks: fail loudly instead of reporting impossible numbers
    if not 0 <= metrics["bounce_rate_pct"] <= 100:
        raise ValueError(f"Bounce rate out of range: {metrics['bounce_rate_pct']}")
    if metrics["avg_session_time_s"] < 0:
        raise ValueError("Negative average session time")

    device_metrics = sessions.groupby("device_type").agg(
        sessions=("page_views", "count"),                                  # unique sessions, not rows
        bounce_rate_pct=("page_views", lambda x: (x == 1).mean() * 100),
        avg_session_time_s=("duration_s", "mean"),
        pages_per_session=("page_views", "mean"),
        total_page_views=("page_views", "sum"),
    ).round(2)

    return metrics, device_metrics, sessions, report

def plot_engagement_summary(device_metrics, sessions):
    """Four labeled charts summarising engagement by device."""
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))

    device_metrics["bounce_rate_pct"].plot.bar(ax=ax[0, 0], color="indianred", rot=0)
    ax[0, 0].set_title("Bounce Rate by Device")
    ax[0, 0].set_xlabel("Device type"); ax[0, 0].set_ylabel("Bounce rate (%)"); ax[0, 0].set_ylim(0, 100)

    (device_metrics["avg_session_time_s"] / 60).plot.bar(ax=ax[0, 1], color="steelblue", rot=0)
    ax[0, 1].set_title("Average Session Time by Device")
    ax[0, 1].set_xlabel("Device type"); ax[0, 1].set_ylabel("Minutes")

    device_metrics["sessions"].plot.bar(ax=ax[1, 0], color="seagreen", rot=0)
    ax[1, 0].set_title("Sessions by Device")
    ax[1, 0].set_xlabel("Device type"); ax[1, 0].set_ylabel("Number of sessions")

    ax[1, 1].hist(sessions["duration_s"] / 60, bins=30, color="slategray", edgecolor="white")
    ax[1, 1].set_title("Distribution of Session Duration")
    ax[1, 1].set_xlabel("Minutes"); ax[1, 1].set_ylabel("Number of sessions")

    plt.tight_layout()
    plt.show()

# ---- Run the pipeline ----
metrics, device_metrics, sessions, report = analyze_user_engagement(logs_df)

print("Validation report:", report)
print()
for k, v in metrics.items():
    print(f"{k}: {v:,.2f}")
display(device_metrics)
plot_engagement_summary(device_metrics, sessions)

## 5. Discussion Answers
*Written for my version of the prompts. Adjust to match your own outputs.*

**1. How did different prompts approach error identification?**
A vague prompt ("fix my code") tends to patch the symptoms, for example clipping negative times to zero, and leaves the cause in place. Pasting the code, describing the symptoms and asking the model to explain each bug led to the root causes:
- `bounce_rate` averages row-level True/False values, so it is not a session bounce rate (it came out at about 0.99 on the sample data).
- `avg_session_time` returns a Series of per-session sums instead of one average, and the raw `duration` values include negatives.
- `pages_per_session` also returns one value per session instead of a single number.
- In `device_metrics`, `'session_id': 'count'` counts rows rather than unique sessions, and `'duration': 'mean'` averages per record instead of per session.
- The data itself had duplicates, missing devices, invalid page views and bad durations.

**2. What validation methods were suggested?**
A required-column check, duplicate removal, dropping rows with a missing session ID or device, numeric coercion with `errors="coerce"`, `page_views >= 1`, duration limited to between 0 and 24 hours, a validation report of removed rows by reason, and sanity checks that stop the function if bounce rate is outside 0-100% or average session time is negative.

**3. How was time handling improved?**
The original summed a raw `duration` column without checking it. The final version calculates duration from the timestamps with `pd.to_datetime(errors="coerce", utc=True)` and `total_seconds()`, so mixed time zones and bad dates are handled. It uses the logged duration only when the timestamps can't be parsed, rejects negative and over-24-hour values, and sums to the session level before averaging. Results are kept in seconds and shown in minutes on the charts.

---
## Scenario 3: Customer Segmentation Query

**Scenario:** The product team needs to segment customers based on their purchasing behavior for a new feature rollout.

**Database schema:**
- `user_activity` (user_id, last_login_date, feature_usage_count, account_type)
- `transactions` (transaction_id, user_id, transaction_date, amount, platform)
- `user_preferences` (user_id, communication_preference, interface_theme, notification_settings)

**Task:** Create a SQL query to identify:
- Active users (logged in during the last 30 days)
- High-value customers (top 20% by spending)
- User preference trends for the identified customers

### Sample database

The notebook runs the SQL with Python's built-in `sqlite3` on a small in-memory database that I generate for testing. It includes some users with no login date, no transactions or no preferences row, so the edge cases are covered. Replace the connection with your own database to run it on real data.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

rng = np.random.default_rng(11)
today = pd.Timestamp.today().normalize()
AS_OF = today.strftime("%Y-%m-%d")        # "today" as a parameter (not hardcoded)
n_users = 500
users = np.arange(1, n_users + 1)

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE user_activity (user_id INTEGER PRIMARY KEY, last_login_date TEXT,
                            feature_usage_count INTEGER, account_type TEXT);
CREATE TABLE transactions (transaction_id INTEGER PRIMARY KEY, user_id INTEGER,
                           transaction_date TEXT, amount REAL, platform TEXT);
CREATE TABLE user_preferences (user_id INTEGER PRIMARY KEY, communication_preference TEXT,
                               interface_theme TEXT, notification_settings TEXT);
""")

# user_activity (dates stored as ISO text; 15 users have no login date)
logins = [(today - pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in rng.integers(0, 120, n_users)]
activity = pd.DataFrame({
    "user_id": users,
    "last_login_date": logins,
    "feature_usage_count": rng.integers(0, 200, n_users),
    "account_type": rng.choice(["Free", "Premium", "Enterprise"], n_users, p=[0.6, 0.3, 0.1]),
})
activity.loc[rng.choice(n_users, 15, replace=False), "last_login_date"] = None
activity.to_sql("user_activity", conn, if_exists="append", index=False)

# transactions (0-15 per user, so some users never bought anything)
tx, tid = [], 1
for u in users:
    for _ in range(int(rng.integers(0, 16))):
        d = (today - pd.Timedelta(days=int(rng.integers(0, 365)))).strftime("%Y-%m-%d")
        tx.append((tid, int(u), d, round(float(rng.lognormal(3.5, 0.9)), 2), str(rng.choice(["web", "ios", "android"]))))
        tid += 1
pd.DataFrame(tx, columns=["transaction_id", "user_id", "transaction_date", "amount", "platform"]) \
  .to_sql("transactions", conn, if_exists="append", index=False)

# user_preferences (about 10% of users have no row)
pref_users = rng.choice(users, int(n_users * 0.9), replace=False)
pd.DataFrame({
    "user_id": pref_users,
    "communication_preference": rng.choice(["Email", "SMS", "Push", "None"], len(pref_users), p=[0.4, 0.2, 0.3, 0.1]),
    "interface_theme": rng.choice(["Light", "Dark", "System"], len(pref_users), p=[0.35, 0.45, 0.2]),
    "notification_settings": rng.choice(["All", "Important only", "Off"], len(pref_users), p=[0.4, 0.4, 0.2]),
}).to_sql("user_preferences", conn, if_exists="append", index=False)

for t in ["user_activity", "transactions", "user_preferences"]:
    print(t, conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "rows")

## 1. Your Prompt

I'm a data analyst on a product team that needs to segment customers for a new feature rollout. My database (SQLite) has three tables: `user_activity` (user_id, last_login_date, feature_usage_count, account_type), `transactions` (transaction_id, user_id, transaction_date, amount, platform) and `user_preferences` (user_id, communication_preference, interface_theme, notification_settings). Dates are stored as ISO text (YYYY-MM-DD).

Write a SQL query that:

1. Finds active users (logged in during the last 30 days).
2. Keeps only high-value customers (top 20% by total spending).
3. Shows preference trends (communication, theme, notifications) for those customers.

Requirements: explain how you calculate the percentile, use CTEs for readability, handle users with no transactions or no preferences row, and suggest indexes.

**Why I set it up this way:** I gave the exact schema so the joins would be correct, and I named the SQL dialect because percentile and date functions differ between databases. I wrote out "top 20% by total spending" and asked for the percentile method to be explained so the ranking definition is visible, not hidden in the query. I asked for NULL handling up front because missing logins and missing preference rows are common in real data.

## 2. Generated Code (first draft)

In [ ]:
DRAFT_SQL = """
SELECT u.user_id, u.account_type, ROUND(s.total_spend, 2) AS total_spend,
       p.communication_preference, p.interface_theme, p.notification_settings
FROM user_activity u
JOIN (
    SELECT user_id, SUM(amount) AS total_spend,
           NTILE(5) OVER (ORDER BY SUM(amount) DESC) AS quintile
    FROM transactions
    GROUP BY user_id
) s ON s.user_id = u.user_id
JOIN user_preferences p ON p.user_id = u.user_id
WHERE s.quintile = 1
  AND u.last_login_date >= date('now', '-30 days')
ORDER BY s.total_spend DESC;
"""

draft = pd.read_sql_query(DRAFT_SQL, conn)
print("Draft result:", len(draft), "customers")
display(draft.head())

## 3. Follow-up Prompts

1. "Rank spending only among active users, and handle ties properly. NTILE can put tied spenders in different groups, so use a cumulative distribution instead."
2. "Rewrite the nested subqueries as readable CTEs and pass the date as a parameter instead of using `date('now')`."
3. "Use a LEFT JOIN so customers without a preferences row aren't dropped, and aggregate the preference trends (count and % of the segment) for each preference column."
4. "Suggest indexes and show the query plan."

**Hurdles:** the first draft ranked everyone who ever bought something and only then filtered to active users, so the segment was not really the top 20% of active customers. Its inner join to `user_preferences` also silently dropped customers with no preferences row. I also had to decide what "top 20%" is measured against (I used active users with at least one transaction), and SQLite has no `PERCENTILE_CONT`, so I used `CUME_DIST()`.

## 4. Final Solution (after refinements)

In [ ]:
# Shared CTEs: active users -> spend -> percentile rank -> top 20% -> preferences
SEGMENT_CTE = """
WITH active_users AS (
    -- Date filter: compare the raw column to a parameter so an index can be used
    SELECT user_id, account_type, feature_usage_count
    FROM user_activity
    WHERE last_login_date >= date(:as_of, '-30 days')
      AND last_login_date <= :as_of
),
user_spend AS (
    -- Total spend per active user (users with no transactions drop out of the ranking)
    SELECT a.user_id, a.account_type, a.feature_usage_count, SUM(t.amount) AS total_spend
    FROM active_users a
    JOIN transactions t ON t.user_id = a.user_id
    GROUP BY a.user_id, a.account_type, a.feature_usage_count
),
ranked AS (
    -- CUME_DIST = share of users whose spend is >= this user's spend (ties stay together)
    SELECT *, CUME_DIST() OVER (ORDER BY total_spend DESC) AS spend_cume_dist
    FROM user_spend
),
high_value AS (
    SELECT * FROM ranked WHERE spend_cume_dist <= 0.20      -- top 20%
),
segment AS (
    -- LEFT JOIN + COALESCE keeps customers who have no preferences row
    SELECT h.user_id, h.account_type, h.feature_usage_count,
           ROUND(h.total_spend, 2) AS total_spend,
           COALESCE(p.communication_preference, 'Not set') AS communication_preference,
           COALESCE(p.interface_theme, 'Not set')          AS interface_theme,
           COALESCE(p.notification_settings, 'Not set')    AS notification_settings
    FROM high_value h
    LEFT JOIN user_preferences p ON p.user_id = h.user_id
)
"""

# Query A: the identified customers
CUSTOMERS_SQL = SEGMENT_CTE + "SELECT * FROM segment ORDER BY total_spend DESC;"

# Query B: preference trends for those customers
TRENDS_SQL = SEGMENT_CTE + """
SELECT 'communication_preference' AS preference_type, communication_preference AS preference_value,
       COUNT(*) AS users, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM segment), 1) AS pct_of_segment
FROM segment GROUP BY communication_preference
UNION ALL
SELECT 'interface_theme', interface_theme,
       COUNT(*), ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM segment), 1)
FROM segment GROUP BY interface_theme
UNION ALL
SELECT 'notification_settings', notification_settings,
       COUNT(*), ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM segment), 1)
FROM segment GROUP BY notification_settings
ORDER BY preference_type, users DESC;
"""

# Indexes that support the date filter, the join and the SUM
conn.executescript("""
CREATE INDEX IF NOT EXISTS idx_activity_last_login ON user_activity(last_login_date);
CREATE INDEX IF NOT EXISTS idx_tx_user_amount ON transactions(user_id, amount);
""")

params = {"as_of": AS_OF}
customers = pd.read_sql_query(CUSTOMERS_SQL, conn, params=params)
trends = pd.read_sql_query(TRENDS_SQL, conn, params=params)

# Sanity check: the segment should be about 20% of active users who have spend
counts = pd.read_sql_query(
    SEGMENT_CTE + "SELECT (SELECT COUNT(*) FROM active_users) AS active_users, "
                  "(SELECT COUNT(*) FROM user_spend) AS active_spenders, "
                  "(SELECT COUNT(*) FROM segment) AS high_value_customers;", conn, params=params)
display(counts)

print("High-value active customers (top 10 by spend):")
display(customers.head(10))

print("Preference trends for the segment:")
display(trends)

print("Query plan (shows which indexes the planner uses):")
for row in conn.execute("EXPLAIN QUERY PLAN " + CUSTOMERS_SQL, params):
    print(" ", row[-1])

## 5. Discussion Answers
*Written for my version of the prompts. Adjust to match your own outputs.*

**1. How were percentile calculations handled?**
The first draft used `NTILE(5)` and kept the first group. That splits users into five equal-size groups, so tied spenders can land in different groups, and it ranked everyone who had ever bought something before the active filter was applied. The final version filters to active users first, then uses `CUME_DIST() OVER (ORDER BY total_spend DESC)` and keeps users with a value of 0.20 or less. This keeps ties together and gives the top 20% of active spenders. SQLite has no `PERCENTILE_CONT`; in PostgreSQL you could instead compute the 80th percentile threshold with `PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY total_spend)` and keep users at or above it.

**2. What approaches to date filtering were used?**
The draft used `date('now', '-30 days')`, which depends on the database clock. The final query passes the date as a parameter (`:as_of`), so results are repeatable and testable. It compares the raw `last_login_date` column with `>=` instead of wrapping the column in a function, so an index can be used, and it also caps the range at the `:as_of` date to ignore future dates. Users with a NULL login date are excluded automatically because the comparison is not true. Other databases use `CURRENT_DATE - INTERVAL '30 days'` (PostgreSQL) or `DATE_SUB(CURDATE(), INTERVAL 30 DAY)` (MySQL).

**3. How was the query optimized (CTE, subquery, etc.)?**
- CTEs replaced the nested subqueries for readability. Most engines plan them like subqueries, so the gain is clarity rather than speed.
- Active users are filtered first, so spend and ranking are computed on fewer rows.
- Transactions are aggregated once, and preferences are joined last, only for the small high-value segment.
- A `LEFT JOIN` with `COALESCE` keeps customers with no preferences row.
- Indexes on `user_activity(last_login_date)` and `transactions(user_id, amount)` support the date filter and the join/SUM, and `EXPLAIN QUERY PLAN` shows which ones the planner actually uses. On this tiny sample it picks the covering index on `transactions`; on a large table the login-date index is what should let the engine skip old logins instead of scanning every row.